## Step 0 — Gradient Checker

In [106]:
from __future__ import annotations  # lazy annotations: allows tuple[...] / X | None on older kernels

from typing import Callable
import numpy as np

In [107]:
def numeric_gradient(
    scalar_function: Callable[[np.ndarray], float],
    feature_values: np.ndarray,
    h: float = 1e-5
) -> np.ndarray:
    gradient = np.zeros_like(feature_values, dtype=float)

    for i in range(feature_values.size):
        mask = np.zeros_like(feature_values, dtype=float)
        mask.flat[i] = h
        gradient.flat[i] = (scalar_function(feature_values + mask) - scalar_function(feature_values - mask)) / (2 * h)
    return gradient

def stable_softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

### Tests — `numeric_gradient`

In [108]:
def _check(name, got, want, atol=1e-6):
    ok = np.allclose(got, want, atol=atol)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", got)
        print("   want:", want)

# 1. f = sum(x^2)  ->  grad = 2x        (vector input, nonlinear)
x = np.array([3.0, -1.0, 0.5])
_check("sum(x^2) grad == 2x", numeric_gradient(lambda v: np.sum(v**2), x), 2 * x)

# 2. f = sum(c*x)  ->  grad = c         (linear -> constant gradient)
c = np.array([2.0, -3.0, 0.7])
_check("sum(c*x) grad == c", numeric_gradient(lambda v: np.sum(c * v), np.zeros(3)), c)

# 3. matrix input -> gradient keeps the matrix shape
W = np.arange(6, dtype=float).reshape(2, 3)
g = numeric_gradient(lambda M: np.sum(M**2), W)
_check("matrix grad == 2W", g, 2 * W)
_check("matrix grad keeps shape", np.array(g.shape), np.array(W.shape))

# 4. f = sum(sin x) -> grad = cos x     (check vs analytic nonlinear)
x = np.array([0.1, 0.7, -1.2, 2.0])
_check("sum(sin x) grad == cos x", numeric_gradient(lambda v: np.sum(np.sin(v)), x), np.cos(x))

[PASS] sum(x^2) grad == 2x
[PASS] sum(c*x) grad == c
[PASS] matrix grad == 2W
[PASS] matrix grad keeps shape
[PASS] sum(sin x) grad == cos x


### Tests — `stable_softmax`

In [109]:
# reuses _check from the numeric_gradient test cell above
x = np.array([2.0, 1.0, 0.1])
p = stable_softmax(x)

_check("probs sum to 1", p.sum(), 1.0)
_check("all in (0, 1)", np.all((p > 0) & (p < 1)), True)

# matches the naive definition on small, safe inputs
naive = np.exp(x) / np.sum(np.exp(x))
_check("matches naive softmax", p, naive)

# stability + shift-invariance: huge logits stay finite and give the same result
big = stable_softmax(x + 1000)
_check("finite on x + 1000", np.all(np.isfinite(big)), True)
_check("shift-invariant (== p)", big, p)

# monotonic: largest logit keeps the largest probability
_check("argmax preserved", np.argmax(p), np.argmax(x))

[PASS] probs sum to 1
[PASS] all in (0, 1)
[PASS] matches naive softmax
[PASS] finite on x + 1000
[PASS] shift-invariant (== p)
[PASS] argmax preserved


## Step 1 — Linear Layer

In [110]:
class Linear:
    """Fully-connected layer:  Y = X @ W + b

    Shapes:
        X : (batch, n_in)     activations from the previous layer
        W : (n_in, n_out)
        b : (n_out,)
        Y : (batch, n_out)
    """

    def __init__(self, n_in: int, n_out: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        
        # simple init: standard normal weights, zero bias
        self.W: np.ndarray = rng.standard_normal((n_in, n_out))
        self.b: np.ndarray = np.zeros(n_out)
            
        # caches / gradient buffers (filled during forward/backward)
        self.X: np.ndarray | None = None   # previous layer's activations, saved for backward
        self.d_w: np.ndarray | None = None
        self.d_b: np.ndarray | None = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (batch, n_in) -> Y: (batch, n_out)."""
        self.X = X                     # cache X: backward needs it to compute d_w
        return X @ self.W + self.b     # b (n_out,) broadcasts across every row

    def backward(self, d_y: np.ndarray) -> np.ndarray:
        """d_y: (batch, n_out) upstream gradient ∂L/∂Y. Returns d_x: (batch, n_in)."""
        
        # d_w: chain rule + sum over the batch -> X.T @ d_y.
        #   local derivative ∂Y/∂W is X; a weight is reused across all samples,
        #   so the matmul sums those per-sample contributions. Shape (n_in, n_out).
        self.d_w = self.X.T @ d_y

        # d_b: local derivative ∂Y/∂b is 1, so just sum d_y over the batch axis.
        #   Shape (n_out,) -- one gradient per bias.
        self.d_b = d_y.sum(axis=0)

        # d_x: gradient to hand back to the previous layer (becomes its d_y).
        #   local derivative ∂Y/∂X is W, so d_x = d_y @ W.T. Shape (batch, n_in).
        d_x = d_y @ self.W.T
        
        return d_x

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(W, d_w), (b, d_b)].

        The optimizer reads this each step: values are mutated in place, gradients
        are re-fetched (backward rebinds d_w/d_b to new arrays every call).
        """
        return [(self.W, self.d_w), (self.b, self.d_b)]

    def zero_grad(self) -> None:
        """Reset gradient buffers (called after each optimizer step)."""
        self.d_w = None
        self.d_b = None

### Test — `Linear` gradient check

In [111]:
# Uses numeric_gradient + _check from Step 0.
# Trick: wrap the layer in a SCALAR loss  L = sum(Y * d_y)  (so ∂L/∂Y = d_y),
# then numeric_gradient of L w.r.t. each of W, b, X must match backward().
def gradient_check_linear(n_in=4, n_out=3, batch=5, seed=1):
    rng = np.random.default_rng(seed)
    layer = Linear(n_in, n_out)
    X  = rng.standard_normal((batch, n_in))
    d_y = rng.standard_normal((batch, n_out))     # random upstream (not all-ones)

    W0, b0 = layer.W.copy(), layer.b.copy()

    # analytic gradients from the layer's own backward
    layer.forward(X)
    d_x = layer.backward(d_y)
    d_w_analytic, d_b_analytic = layer.d_w.copy(), layer.d_b.copy()

    # numeric d_w: vary W, hold X/b fixed
    def loss_W(W_flat):
        layer.W = W_flat.reshape(W0.shape)
        return np.sum(layer.forward(X) * d_y)
    d_w_numeric = numeric_gradient(loss_W, W0.copy()); layer.W = W0.copy()

    # numeric d_b: vary b
    def loss_b(b_flat):
        layer.b = b_flat.reshape(b0.shape)
        return np.sum(layer.forward(X) * d_y)
    d_b_numeric = numeric_gradient(loss_b, b0.copy()); layer.b = b0.copy()

    # numeric d_x: vary X, params at originals
    def loss_X(X_flat):
        return np.sum(layer.forward(X_flat.reshape(X.shape)) * d_y)
    d_x_numeric = numeric_gradient(loss_X, X.copy())

    _check("∂Linear/∂W", d_w_analytic, d_w_numeric)
    _check("∂Linear/∂b", d_b_analytic, d_b_numeric)
    _check("∂Linear/∂X", d_x,   d_x_numeric)

gradient_check_linear()

[PASS] dLinear/dW
[PASS] dLinear/db
[PASS] dLinear/dX


## Step 2 — Cross-Entropy Loss

In [112]:
def cross_entropy(
    raw_class_scores: np.ndarray,
    correct_class_indices: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Softmax cross-entropy for a batch of classification examples.

    Args:
        raw_class_scores      : (number_of_examples_in_batch, number_of_classes)
                              raw scores (logits), one row per example
        correct_class_indices : (number_of_examples_in_batch,)
                              correct class index for each example

    Returns:
        mean_loss          : float        mean cross-entropy over the batch
        gradient_wrt_scores : np.ndarray   (number_of_examples_in_batch, number_of_classes)
                            ∂L/∂scores = (softmax - onehot) / number_of_examples_in_batch
    """
    number_of_examples_in_batch: int = raw_class_scores.shape[0]
    example_rows: np.ndarray = np.arange(number_of_examples_in_batch)   # [0, 1, ..., batch-1], to index each row

    # --- stable softmax, per row (subtract each row's max -> no overflow) ---
    stabilized_scores: np.ndarray = raw_class_scores - raw_class_scores.max(axis=1, keepdims=True)
    exponentiated_scores: np.ndarray = np.exp(stabilized_scores)
    class_probabilities: np.ndarray = exponentiated_scores / exponentiated_scores.sum(axis=1, keepdims=True)

    # --- loss: -log(prob of the correct class), averaged over the batch ---
    probability_of_correct_class: np.ndarray = class_probabilities[example_rows, correct_class_indices]
    negative_log_prob_of_correct_class: np.ndarray = -np.log(probability_of_correct_class)   # (batch,)
    mean_loss: float = float(negative_log_prob_of_correct_class.mean())

    # --- gradient: the clean combined form  softmax - onehot  ---
    #   start from probabilities, subtract 1 at each row's correct class, average over batch
    gradient_wrt_scores: np.ndarray = class_probabilities.copy()
    gradient_wrt_scores[example_rows, correct_class_indices] -= 1
    gradient_wrt_scores /= number_of_examples_in_batch

    return mean_loss, gradient_wrt_scores

### Test — `cross_entropy` gradient check

In [113]:
# The combined gradient softmax - onehot must match finite differences of the loss.
def gradient_check_cross_entropy(number_of_examples_in_batch=4, number_of_classes=5, seed=1):
    random_generator = np.random.default_rng(seed)
    random_scores = random_generator.standard_normal((number_of_examples_in_batch, number_of_classes))
    correct_class_indices = random_generator.integers(0, number_of_classes, size=number_of_examples_in_batch)

    analytic_loss, analytic_gradient = cross_entropy(random_scores, correct_class_indices)

    # numeric gradient: loss as a scalar function of the scores
    def loss_as_function_of_scores(flattened_scores):
        loss_value, _ = cross_entropy(flattened_scores.reshape(random_scores.shape), correct_class_indices)
        return loss_value
    numeric_gradient_value = numeric_gradient(loss_as_function_of_scores, random_scores.copy())

    _check("∂CE/∂scores", analytic_gradient, numeric_gradient_value)

gradient_check_cross_entropy()

[PASS] dCE/dScores


## Step 2 — Adam Optimizer

In [114]:
class Adam:
    """Adam optimizer: updates every parameter of the given layers in place.

    Holds per-parameter state (first/second moment estimates) and shared state
    (timestep, learning rate, betas, epsilon). Each parameter's update is fully
    independent -- the optimizer just loops over all of them.

    Design:
        - references the layers, so it can reach every parameter
        - reads gradients FRESH each step via layer.parameters()
          (backward rebinds d_w/d_b to new arrays every call)
        - updates values IN PLACE (value -= ...), so the layer's W / b actually change
    """

    def __init__(
        self,
        layers: list,
        lr: float = 1e-3,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
    ) -> None:
        self.layers: list = layers
        self.lr: float = lr
        self.beta1: float = beta1
        self.beta2: float = beta2
        self.eps: float = eps
        self.time_step: int = 0

        # one moment buffer per parameter, in the stable order parameters() yields.
        # sized from the VALUES (grads may still be None before the first backward).
        self.first_moment_estimates: list[np.ndarray] = []
        self.second_moment_estimates: list[np.ndarray] = []
        for layer in self.layers:
            for (parameter_value, _parameter_gradient) in layer.parameters():
                self.first_moment_estimates.append(np.zeros_like(parameter_value))
                self.second_moment_estimates.append(np.zeros_like(parameter_value))

    def step(self) -> None:
        """Apply one Adam update to every parameter, using the current gradients."""
        self.time_step += 1
        bias_correction_first: float = 1.0 - self.beta1 ** self.time_step
        bias_correction_second: float = 1.0 - self.beta2 ** self.time_step

        parameter_index: int = 0
        for layer in self.layers:
            for (parameter_value, parameter_gradient) in layer.parameters():   # fresh grads
                first_moment: np.ndarray = self.first_moment_estimates[parameter_index]
                second_moment: np.ndarray = self.second_moment_estimates[parameter_index]

                # update biased moment estimates (in place, so buffer identity is kept)
                first_moment *= self.beta1
                first_moment += (1.0 - self.beta1) * parameter_gradient
                second_moment *= self.beta2
                second_moment += (1.0 - self.beta2) * (parameter_gradient ** 2)

                # bias-corrected estimates (early steps would otherwise be too small)
                corrected_first_moment: np.ndarray = first_moment / bias_correction_first
                corrected_second_moment: np.ndarray = second_moment / bias_correction_second

                # in-place parameter update -> mutates the layer's W / b
                parameter_value -= self.lr * corrected_first_moment / (np.sqrt(corrected_second_moment) + self.eps)

                parameter_index += 1

    def zero_grad(self) -> None:
        """Reset every layer's gradient buffers after a step."""
        for layer in self.layers:
            layer.zero_grad()

### Test — `Adam` reduces loss on a learnable task

In [115]:
# Full training loop on a linearly-separable task: a single Linear + cross_entropy,
# optimized by Adam. Labels come from a linear rule, so the model can actually fit them.
def test_adam_reduces_loss(seed=0):
    random_generator = np.random.default_rng(seed)
    number_of_examples, number_of_features, number_of_classes = 64, 5, 3

    inputs = random_generator.standard_normal((number_of_examples, number_of_features))
    true_weights = random_generator.standard_normal((number_of_features, number_of_classes))
    correct_class_indices = np.argmax(inputs @ true_weights, axis=1)   # learnable labels

    layer = Linear(number_of_features, number_of_classes)
    optimizer = Adam([layer], lr=0.1)

    weights_before = layer.W.copy()
    first_loss = None
    last_loss = None
    for step_index in range(300):
        raw_class_scores = layer.forward(inputs)
        loss, gradient_wrt_scores = cross_entropy(raw_class_scores, correct_class_indices)
        layer.backward(gradient_wrt_scores)
        optimizer.step()
        optimizer.zero_grad()
        if step_index == 0:
            first_loss = loss
        last_loss = loss

    predictions = np.argmax(layer.forward(inputs), axis=1)
    accuracy = float((predictions == correct_class_indices).mean())

    _check("Adam drives loss down", last_loss < first_loss * 0.5, True)
    _check("Adam updated the weights in place", np.any(layer.W != weights_before), True)
    _check("fits the learnable task (acc > 0.9)", accuracy > 0.9, True)
    print(f"   loss: {first_loss:.3f} -> {last_loss:.3f}   accuracy: {accuracy:.2f}")

test_adam_reduces_loss()

[PASS] Adam drives loss down
[PASS] Adam updated the weights in place
[PASS] fits the learnable task (acc > 0.9)
   loss: 2.672 -> 0.065   accuracy: 0.98


## CharTokenizer — text ↔ tokens

Character-level tokenizer: converts text to integer token ids and back. Not a Layer — it has
no parameters and the optimizer never touches it. Vocabulary = special tokens
(`<pad>`, `<bos>`, `<eos>`) + every unique character in the training text.

In [116]:
class CharTokenizer:
    """Character-level tokenizer: text <-> list of integer token ids.

    Vocabulary = special tokens + every unique character in the training text.
    Not a Layer: no parameters, never seen by the optimizer.
    """

    PAD_TOKEN: str = "<pad>"
    BOS_TOKEN: str = "<bos>"   # beginning of sequence
    EOS_TOKEN: str = "<eos>"   # end of sequence

    def __init__(self, training_text: str) -> None:
        special_tokens: list[str] = [self.PAD_TOKEN, self.BOS_TOKEN, self.EOS_TOKEN]
        unique_characters: list[str] = sorted(set(training_text))
        # id -> token (list index is the id); token -> id (inverse map)
        self.id_to_token: list[str] = special_tokens + unique_characters
        self.token_to_id: dict[str, int] = {token: token_id for token_id, token in enumerate(self.id_to_token)}

    @property
    def vocab_size(self) -> int:
        return len(self.id_to_token)

    @property
    def pad_id(self) -> int:
        return self.token_to_id[self.PAD_TOKEN]

    @property
    def bos_id(self) -> int:
        return self.token_to_id[self.BOS_TOKEN]

    @property
    def eos_id(self) -> int:
        return self.token_to_id[self.EOS_TOKEN]

    def encode(self, text: str, add_specials: bool = True) -> list[int]:
        """text -> token ids. If add_specials, wrap as <bos> ... <eos>."""
        token_ids: list[int] = [self.token_to_id[character] for character in text]
        if add_specials:
            token_ids = [self.bos_id] + token_ids + [self.eos_id]
        return token_ids

    def decode(self, token_ids: list[int], skip_specials: bool = True) -> str:
        """token ids -> text. If skip_specials, drop <pad>/<bos>/<eos>."""
        special_token_ids: set[int] = {self.pad_id, self.bos_id, self.eos_id}
        return "".join(
            self.id_to_token[token_id]
            for token_id in token_ids
            if not (skip_specials and token_id in special_token_ids)
        )

### Test — `CharTokenizer` round-trip

In [117]:
# _check uses np.allclose (numeric); add a plain-equality check for strings/ints/lists.
def _check_eq(name, got, want):
    ok = (got == want)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", repr(got))
        print("   want:", repr(want))

def test_char_tokenizer():
    tokenizer = CharTokenizer("a dog runs")

    # vocab = 3 special tokens + unique characters of the training text
    unique_character_count = len(set("a dog runs"))
    _check_eq("vocab size", tokenizer.vocab_size, 3 + unique_character_count)

    # round-trip: decode(encode(text)) recovers the text (specials skipped on decode)
    for text in ["a dog", "runs", "a", "dog runs"]:
        _check_eq(f"round-trip {text!r}", tokenizer.decode(tokenizer.encode(text)), text)

    # encode wraps with <bos> ... <eos>
    token_ids = tokenizer.encode("a")
    _check_eq("starts with <bos>", token_ids[0], tokenizer.bos_id)
    _check_eq("ends with <eos>", token_ids[-1], tokenizer.eos_id)

    # decode can keep the special tokens when asked
    _check_eq("decode keeps specials", tokenizer.decode(token_ids, skip_specials=False), "<bos>a<eos>")

    # add_specials=False encodes just the characters
    _check_eq("no specials when off", tokenizer.encode("a", add_specials=False), [tokenizer.token_to_id["a"]])

test_char_tokenizer()

[PASS] vocab size
[PASS] round-trip 'a dog'
[PASS] round-trip 'runs'
[PASS] round-trip 'a'
[PASS] round-trip 'dog runs'
[PASS] starts with <bos>
[PASS] ends with <eos>
[PASS] decode keeps specials
[PASS] no specials when off


## Embedding — token id → learned vector

A Layer with **one** parameter: the table `(vocab_size, d)`, one row per token.
- **forward** is a row lookup: `table[ids]` (no computation, just indexing).
- **backward** is a **scatter-add**: each used row receives the gradient of every position
  that used it (duplicate ids accumulate). It returns **`None`** — integer ids aren't
  differentiable, and Embedding is always the first layer.

Same `parameters()` contract as `Linear`, so `Adam` updates its table with no special-casing.

In [118]:
class Embedding:
    """Embedding layer: maps integer token ids to learned vectors (a row lookup).

    One trainable parameter: table (vocab_size, embedding_dim), one row per token.
        forward(ids)  -> table[ids]   (rows selected; caches ids)
        backward(d_out) -> None        (scatter-add into d_table; duplicate ids accumulate)

    Returns no input gradient — integer ids aren't differentiable, and Embedding is the
    first layer. Same parameters() contract as Linear, so the optimizer treats it the same.
    """

    def __init__(self, vocab_size: int, embedding_dim: int, seed: int = 0) -> None:
        random_generator = np.random.default_rng(seed)
        self.table: np.ndarray = random_generator.standard_normal((vocab_size, embedding_dim))
        # cache + gradient buffer (filled during forward/backward)
        self.token_ids: np.ndarray | None = None
        self.d_table: np.ndarray | None = None

    def forward(self, token_ids: np.ndarray) -> np.ndarray:
        """token_ids: integer array of shape S -> embeddings of shape S + (embedding_dim,)."""
        self.token_ids = token_ids          # cache ids: backward needs to know which rows were used
        return self.table[token_ids]       # fancy indexing = row lookup

    def backward(self, d_out: np.ndarray) -> None:
        """d_out: ∂L/∂(looked-up rows), same shape as the forward output.

        Scatter-add each position's gradient into the row of its id. np.add.at accumulates
        on repeated ids (plain d_table[ids] = d_out would overwrite duplicates -> wrong).
        Returns None: there is no gradient to pass back to integer ids.
        """
        self.d_table = np.zeros_like(self.table)
        np.add.at(self.d_table, self.token_ids, d_out)
        return None

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(table, d_table)]."""
        return [(self.table, self.d_table)]

    def zero_grad(self) -> None:
        """Reset the gradient buffer (called after each optimizer step)."""
        self.d_table = None

### Test — `Embedding` gradient check (scatter-add)

In [119]:
# Scatter-add backward must match finite differences of the table.
# Use a REPEATED id so the accumulation path is exercised.
def gradient_check_embedding(vocab_size=6, embedding_dim=4, seed=1):
    random_generator = np.random.default_rng(seed)
    embedding = Embedding(vocab_size, embedding_dim)

    token_ids = np.array([2, 0, 4, 2, 2])   # id 2 appears 3x -> tests scatter-add accumulation
    upstream_gradient = random_generator.standard_normal((len(token_ids), embedding_dim))

    # forward shape sanity
    embedded_rows = embedding.forward(token_ids)
    _check_eq("forward shape", embedded_rows.shape, (len(token_ids), embedding_dim))

    # analytic gradient from scatter-add backward
    return_value = embedding.backward(upstream_gradient)
    analytic_gradient = embedding.d_table.copy()
    _check_eq("backward returns None", return_value, None)

    # accumulation: the repeated id's row = sum of the gradients at its occurrences (0, 3, 4)
    _check("scatter-add accumulates", analytic_gradient[2], upstream_gradient[[0, 3, 4]].sum(axis=0))

    # numeric gradient: L = sum(table[ids] * upstream_gradient) as a function of the table
    original_table = embedding.table.copy()
    def loss_as_function_of_table(flattened_table):
        embedding.table = flattened_table.reshape(original_table.shape)
        return np.sum(embedding.forward(token_ids) * upstream_gradient)
    numeric_gradient_value = numeric_gradient(loss_as_function_of_table, original_table.copy())
    embedding.table = original_table

    _check("∂Embedding/∂table", analytic_gradient, numeric_gradient_value)

gradient_check_embedding()

[PASS] forward shape
[PASS] backward returns None
[PASS] scatter-add accumulates
[PASS] dEmbedding/dTable


## Bigram — next token from the current token

The first assembled model, and the integration test for everything so far. It predicts the
next token from **only the current token**: `Embedding(vocab, d) → Linear(d, vocab) → logits`.

- **`forward`** (both phases) — embed the current token, project to next-token logits.
- **`backward`** (training) — push the loss gradient back through Linear, then Embedding.
- **`generate`** (working) — forward only: sample a token, feed it back, until `<eos>`.

`parameters()` / `zero_grad()` just delegate to the two sub-layers, so `Adam([model])` updates
both with no special-casing.

In [120]:
class Bigram:
    """Bigram language model: predicts the next token from ONLY the current token.

    Composition:  Embedding(vocab, d) -> Linear(d, vocab) -> next-token logits.
    forward + backward are used in training; forward alone drives generation.
    """

    def __init__(self, vocab_size: int, embedding_dim: int, seed: int = 0) -> None:
        self.vocab_size: int = vocab_size
        self.embedding: Embedding = Embedding(vocab_size, embedding_dim, seed=seed)
        self.projection: Linear = Linear(embedding_dim, vocab_size, seed=seed + 1)

    def forward(self, current_token_ids: np.ndarray) -> np.ndarray:
        """current_token_ids: (batch,) -> logits (batch, vocab_size)."""
        embedded: np.ndarray = self.embedding.forward(current_token_ids)   # (batch, d)
        logits: np.ndarray = self.projection.forward(embedded)          # (batch, vocab)
        return logits

    def backward(self, gradient_wrt_logits: np.ndarray) -> None:
        """Backprop the loss gradient through projection, then embedding."""
        gradient_wrt_embedded: np.ndarray = self.projection.backward(gradient_wrt_logits)  # (batch, d)
        self.embedding.backward(gradient_wrt_embedded)   # scatter-add into the table; returns None
        return None

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """All (value, gradient) pairs, so one optimizer can update the whole model."""
        return self.embedding.parameters() + self.projection.parameters()

    def zero_grad(self) -> None:
        self.embedding.zero_grad()
        self.projection.zero_grad()

    def generate(
        self,
        tokenizer: CharTokenizer,
        max_new_tokens: int = 100,
        temperature: float = 1.0,
        seed: int = 0,
    ) -> str:
        """Autoregressive generation: start at <bos>, sample tokens until <eos> or the cap."""
        random_generator = np.random.default_rng(seed)
        generated_ids: list[int] = [tokenizer.bos_id]

        for _ in range(max_new_tokens):
            current_token_id: np.ndarray = np.array([generated_ids[-1]])       # (1,)
            logits: np.ndarray = self.forward(current_token_id)[0]           # (vocab,)
            probabilities: np.ndarray = stable_softmax(logits / temperature)
            probabilities = probabilities / probabilities.sum()            # guard against rounding
            next_token_id: int = int(random_generator.choice(self.vocab_size, p=probabilities))
            generated_ids.append(next_token_id)
            if next_token_id == tokenizer.eos_id:
                break

        return tokenizer.decode(generated_ids)

### Train + generate — first text from a from-scratch LM

Train on a small structured corpus with the full stack (`Embedding → Linear → cross_entropy →
Adam`) on shift-by-one pairs, then generate. Success = loss beats the uniform baseline
`ln(vocab)` and the output forms letter patterns.

In [121]:
def train_bigram(
    model: Bigram,
    tokenizer: CharTokenizer,
    text: str,
    steps: int = 500,
    lr: float = 0.1,
) -> list[float]:
    """Full-batch training on shift-by-one pairs. Returns the loss at each step."""
    token_ids: np.ndarray = np.array(tokenizer.encode(text))   # includes <bos> ... <eos>
    current_token_ids: np.ndarray = token_ids[:-1]               # inputs
    next_token_ids: np.ndarray = token_ids[1:]                   # targets (shifted by one)

    optimizer = Adam([model], lr=lr)
    loss_history: list[float] = []
    for _ in range(steps):
        logits = model.forward(current_token_ids)
        loss, gradient_wrt_logits = cross_entropy(logits, next_token_ids)
        model.backward(gradient_wrt_logits)
        optimizer.step()
        optimizer.zero_grad()
        loss_history.append(loss)
    return loss_history


def test_bigram():
    corpus_text = "the quick brown fox jumps over the lazy dog. " * 30
    tokenizer = CharTokenizer(corpus_text)
    model = Bigram(tokenizer.vocab_size, embedding_dim=32)

    loss_history = train_bigram(model, tokenizer, corpus_text, steps=500, lr=0.1)

    uniform_baseline = float(np.log(tokenizer.vocab_size))   # loss of random guessing
    _check("loss decreased", loss_history[-1] < loss_history[0], True)
    _check("beats uniform baseline ln(vocab)", loss_history[-1] < uniform_baseline, True)
    print(f"   loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}   (uniform baseline {uniform_baseline:.3f})")

    sample = model.generate(tokenizer, max_new_tokens=60, temperature=0.8, seed=1)
    print("   sample:", repr(sample))

test_bigram()

[PASS] loss decreased
[PASS] beats uniform baseline ln(vocab)
   loss: 12.641 -> 0.639   (uniform baseline 3.434)
   sample: 'ther juick ove lazy quick overog. ox lazy ove lazy the brox '


## CausalSelfAttention — mixing information across positions

The heart of the model. Each position builds a **query**, compares it against every **key**,
and takes a weighted average of the **values**:

$$\text{attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_{head}}} + \text{mask}\right)V$$

- **Q, K, V** are three `Linear` projections of the same input (reusing our `Linear` layer).
- **`/√d_head`** keeps the scores from growing with dimension, which would saturate softmax.
- The **causal mask** sets future positions to `−∞` *before* softmax, so position `i` can only
  attend to positions `≤ i`. That is what makes the model autoregressive.

Shapes here are **single-sequence** `(T, d_model)` — one sequence at a time. That keeps the
backward pass (the hardest part of the project) simple enough to gradient-check directly;
batching arrives when we assemble the full GPT.